In [ ]:
from dotenv import load_dotenv
load_dotenv()

%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
%gui qt

from copy import deepcopy
import time
import xarray as xr # Assuming you're using this
import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import xarray as xr
# import zarr
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
# import panel as pn
# pn.extension()

import IPython
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pyqtgraph as pg
import phopymnehelper.type_aliases as types
from pypho_timeline.widgets.simple_timeline_widget import SimpleTimelineWidget
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow
from pypho_timeline.__main__ import VideoTrackDatasource

# CustomGraphicsLayoutWidget
from pypho_timeline.xdf_session_discovery import discover_xdf_files_for_timeline
from pypho_timeline.timeline_builder import TimelineBuilder

def get_now_time_str(time_separator='-') -> str:
    return str(time.strftime(f"%Y-%m-%d_%H{time_separator}%m", time.localtime(time.time())))

# Create Qt application
app = pg.mkQApp("pyPhoTimelineTestingNotebookExample")
builder: TimelineBuilder = TimelineBuilder()

In [ ]:
# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 100
n_most_recent_sessions_to_preprocess: int = 15
# n_most_recent_sessions_to_preprocess: int = 8
# n_most_recent_sessions_to_preprocess: int = 5
# n_most_recent_sessions_to_preprocess = None

# enable_video_track: bool = True
enable_video_track: bool = False

# Optional: only include streams whose name matches any of these patterns (regex).
STREAM_ALLOWLIST: Optional[List[str]] = None  # e.g. [r"EEG.*", r"MOTION.*"]

# Optional: exclude streams whose name matches any of these patterns (regex).
# STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X Motion', 'Epoc X eQuality']
# STREAM_BLOCKLIST: Optional[List[str]] = ['Epoc X eQuality', 'VideoRecorderMarkers']
STREAM_BLOCKLIST: Optional[List[str]] = ['VideoRecorderMarkers']


# db_root_path = Path('/content/drive/MyDrive/Databases')
# db_root_path = Path(r'E:/Dropbox (Personal)/Databases') ## APOGEE
db_root_path = Path(r'E:/Dropbox (Personal)/Databases') # WIN10_VM
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs')
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

video_recordings_path: Path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001/Videos')

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_rerun_rrd_parent_export_path = db_root_path.joinpath('AnalysisData/to_rerun_rrd').resolve()
xdf_to_rerun_rrd_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')

# eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed')
# pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION')
# assert pickled_data_path.exists()
xdf_to_exported_EDF_parent_export_path = db_root_path.joinpath('AnalysisData/exported_EDF').resolve()
xdf_to_exported_EDF_parent_export_path.mkdir(exist_ok=True)
# print(f'xdf_to_rerun_rrd_parent_export_path: "{xdf_to_rerun_rrd_parent_export_path.as_posix()}"')


# lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001")
lab_recorder_output_path = db_root_path.joinpath('UnparsedData/LabRecorderStudies/sub-P001')
assert lab_recorder_output_path.exists()

xdf_file_cache_filename: str = f"{get_now_time_str()}_found_xdf_files.csv"
xdf_file_cache_filepath: Path = xdf_to_rerun_rrd_parent_export_path.joinpath(xdf_file_cache_filename).resolve()
print(f'exporting xdf .csv to xdf_file_cache_filepath: "{xdf_file_cache_filepath}..."')
discovery = discover_xdf_files_for_timeline(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, csv_export_path=xdf_file_cache_filepath)
final_xdf_paths: List[Path] = discovery.xdf_paths
print(f'processing len(active_EEG_recording_files): {len(final_xdf_paths)} recording files...')

## OUTPUTS: final_xdf_paths

# ==================================================================================================================================================================================================================================================================================== #
# BEGIN MAIN                                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #

# builder: TimelineBuilder = TimelineBuilder()
# active_video_discovery_dirs: List[Path] = [video_recordings_path] if video_recordings_path.exists() and video_recordings_path.is_dir() else []
active_video_discovery_dirs = []
builder.set_refresh_config(xdf_discovery_dirs=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], n_most_recent=n_most_recent_sessions_to_preprocess, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST, video_discovery_dirs=active_video_discovery_dirs)

In [ ]:
builder.refresh_from_directories()

In [ ]:
timeline: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST)
if timeline is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")


In [ ]:
## Create another independent timeline
timeline_alt: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST)
if timeline_alt is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")


In [ ]:
builder.default_post_timeline_create_display_updates(timeline=timeline_alt)

In [ ]:
STREAM_BLOCKLIST_alt2: Optional[List[str]] = ['Epoc X eQuality', 'VideoRecorderMarkers', 'TextLogger', 'EventBoard', 'log_widget']
## Create another independent timeline
timeline_alt2: SimpleTimelineWidget = builder.build_from_xdf_files(xdf_file_paths=final_xdf_paths, stream_allowlist=STREAM_ALLOWLIST, stream_blocklist=STREAM_BLOCKLIST_alt2)
if timeline_alt2 is None:
    print("WARN: No streams found. Check XDF paths and stream filters.")



In [ ]:
builder.default_post_timeline_create_display_updates(timeline=timeline_alt2)

### Get all timeline tracks

In [ ]:
from phopymnehelper.motion_data import MotionData, MotionDataFrame, BadMotionDataFrame
from phopymnehelper.MNE_helpers import MNEHelpers

all_track_names = timeline.get_all_track_names()
print(f"all_track_names: {all_track_names}")
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')
txt_log_widget, txt_log_renderer, txt_log_ds = timeline.get_track_tuple('LOG_TextLogger')


eeg_spectogram_track_names = ['EEG_Spectrogram_Epoc X_Frontal-L',
    'EEG_Spectrogram_Epoc X_Frontal-R',
    'EEG_Spectrogram_Epoc X_Posterior-L',
    'EEG_Spectrogram_Epoc X_Posterior-R', 'EEG_Spectrogram_Epoc X_All',
]

for a_spectogram_track_name in eeg_spectogram_track_names:
    if a_spectogram_track_name in all_track_names:
        a_spectogram_widget, a_spectogram_track, a_spectogram_ds = timeline.get_track_tuple(a_spectogram_track_name)
        # a_spectogram_track
        a_root_graphics_layout_widget = a_spectogram_widget.getRootGraphicsLayoutWidget()
        a_plot_item = a_spectogram_widget.getRootPlotItem()
        a_plot_item.hideAxis('bottom')
        # a_spectogram_ds.detailed_df


## OUTPUTS: eeg_ds, motion_ds, txt_log_ds

In [ ]:
_out_cal_widget = timeline.add_calendar_navigator()
_out_cal_widget

# Compute Spectograms

In [ ]:
from pypho_timeline.rendering.datasources.specific.eeg import EEGPlotDetailRenderer, EEGTrackDatasource, SpectrogramChannelGroupConfig, compute_multiraw_spectrogram_results


spect_groups = [SpectrogramChannelGroupConfig(name='All', channels=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']),
    SpectrogramChannelGroupConfig(name='Frontal', channels=['AF3', 'F7', 'FC5', 'F3', 'AF4', 'F8', 'FC6', 'F4']),
    SpectrogramChannelGroupConfig(name='Posterior', channels=['T7', 'P7', 'O1', 'T8', 'P8', 'O2']),
    # SpectrogramChannelGroupConfig(name='Left', channels=['AF3', 'F7', 'FC5', 'F3', 'T7', 'P7', 'O1']),
]

_out_spect_ds_list = eeg_ds.add_spectrogram_tracks_for_channel_groups(spectrogram_channel_groups=spect_groups, timeline=timeline, timeline_builder=builder)



In [ ]:
timeline.hide_extra_xaxis_labels_and_axes(enable_hide_extra_track_x_axes=True)

In [ ]:
spec_names = [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_")]
spec_names += [n for n in timeline.track_datasources if n.startswith("EEG_Spectrogram_") and n.endswith("__compare")]
spec_names

In [ ]:
# After: timeline = builder.build_from_xdf_files(...)
assert timeline is not None and len(timeline.get_all_track_names()) > 0, "Build the timeline first."
_names = timeline.get_all_track_names()
_eeg_names = [n for n in _names if n.startswith("EEG_")]
_motion_names = [n for n in _names if n.startswith("MOTION_")]
# if not _eeg_names:
#     raise RuntimeError(f"No EEG_* track found. Names: {_names}")
# eeg_name = _eeg_names[0]
eeg_name = 'EEG_Epoc X' ## HARDCODED
print(f'eeg_name: {eeg_name}')
motion_name = _motion_names[0] if _motion_names else None
eeg_ds = timeline.track_datasources[eeg_name]
eeg_df = eeg_ds.detailed_df.sort_values("t").reset_index(drop=True).copy()
_t_ts = eeg_df["t"]
if pd.api.types.is_datetime64_any_dtype(_t_ts):
    eeg_df["t"] = pd.to_datetime(_t_ts, utc=True, errors="coerce").astype(np.int64) / 1e9
else:
    eeg_df["t"] = pd.to_numeric(_t_ts, errors="coerce")

# # eeg_ds.intervals_df
# from pypho_timeline.rendering.datasources.specific.eeg import _first_nonempty_raw_list_from_dict
# _eeg_raws = _first_nonempty_raw_list_from_dict(eeg_ds.raw_datasets_dict)
# assert len(_eeg_raws) > 0, f"eeg datasource has no raw datasets!"
# eeg_raw = _eeg_raws[0]
# eeg_raw

In [ ]:
eeg_ds

In [ ]:
eeg_raw.annotations

In [ ]:
spec_result = eeg_comps_result["spectogram"]

spec_result


In [ ]:
eeg_ds.compute()

In [ ]:
from pypho_timeline.rendering.datasources.specific.eeg import EEGSpectrogramTrackDatasource
from pypho_timeline.rendering.datasources.stream_to_datasources import default_dock_named_color_scheme_key
from pypho_timeline.docking.dock_display_configs import CustomCyclicColorsDockDisplayConfig, NamedColorScheme
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode
from qtpy import QtWidgets

track_name = "EEG_Spectrogram_all"  # unique
spec_ds = EEGSpectrogramTrackDatasource(
    intervals_df=eeg_ds.intervals_df.copy(),
    spectrogram_result=spec_result,
    custom_datasource_name=track_name,
    group_config=None,
    lab_obj_dict=eeg_ds.lab_obj_dict,
    raw_datasets_dict=eeg_ds.raw_datasets_dict,
    parent=eeg_ds,
)

_scheme_key = default_dock_named_color_scheme_key(track_name)
display_config = CustomCyclicColorsDockDisplayConfig(
    named_color_scheme=NamedColorScheme[_scheme_key],
    showCloseButton=True, showCollapseButton=True, showGroupButton=True, corner_radius="3px",
)
track_widget, root_g, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=track_name,
    dockSize=(500, 80),
    dockAddLocationOpts=["bottom"],
    display_config=display_config,
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA,
)
# same x-axis setup as TimelineBuilder._add_tracks_to_timeline …
plot_item.setYRange(0, 1, padding=0)
plot_item.setLabel("left", track_name)
plot_item.hideAxis("left")
timeline.add_track(spec_ds, name=track_name, plot_item=plot_item)
track_widget.optionsPanel = track_widget.getOptionsPanel()
if hasattr(dock, "updateWidgetsHaveOptionsPanel"):
    dock.updateWidgetsHaveOptionsPanel()
getattr(timeline, "_rebuild_timeline_overview_strip", lambda: None)()

In [ ]:
spec_ds.raw_datasets_dict

# ADHD / theta–delta sleep intrusion (`compute_theta_delta_sleep_intrusion_series`)

Run **after** the timeline is built (previous cell). Resolves `EEG_*` and optional `MOTION_*` tracks, builds `mne.io.RawArray` from full-rate `detailed_df`, aligns motion times to `raw.times`, and stashes `adhd_ctx` for the next cells.

In [ ]:
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import apply_adhd_sleep_intrusion_to_timeline
from phopylslhelper.datetime_helpers import unix_timestamp_to_datetime, datetime_to_unix_timestamp
from datetime import datetime, timedelta

eeg_name: str = timeline.eeg_track_identifier            # or 'EEG_Epoc X'
eeg_ds   = timeline.track_datasources[eeg_name]
eeg_raw  = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)[0]

result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=("theta_delta_sleep_intrusion",))["theta_delta_sleep_intrusion"]
print("session_mean:", result["session_mean_theta_delta"], "valid:", result["n_valid_windows"], "/", result["n_windows"])

td_ratio_widget, td_ratio_track, td_ratio_ds = apply_adhd_sleep_intrusion_to_timeline(timeline, result, eeg_name=eeg_name, eeg_ds=eeg_ds)
detailed_td_ratio_df: pd.DataFrame = td_ratio_ds.detailed_df

In [ ]:
td_ratio_widget.active_plot_target.viewRange() # [[1777458500.7219324, 1777460685.571476], [-0.02, 1.02]]

np.diff([1777458500.7219324, 1777460685.571476])


In [ ]:
x0v, x1v = ref_plot.getViewBox().viewRange()[0]
ratio_plot_item.setXRange(x0v, x1v, padding=0)

ratio_plot_item.setYRange(0, 1, padding=-0.02)

In [ ]:
td_ratio_widget, td_ratio_track, td_ratio_ds = timeline.get_track_tuple("ANALYSIS_theta_delta")
detailed_td_ratio_df: pd.DataFrame = td_ratio_ds.detailed_df
detailed_td_ratio_df

In [ ]:
detailed_td_ratio_df['dt'] = unix_timestamp_to_datetime(detailed_td_ratio_df['t'].to_numpy())
detailed_td_ratio_df

In [ ]:
timeline.reference_datetime

In [ ]:
t0_unix = result.get("t0_unix")
t0_earliest_unix_ts = float(eeg_ds.earliest_unix_timestamp)
t0_earliest_unix_ts
t0_unix - t0_earliest_unix_ts ## earliest unix_timestamp is actually later!
dt0_unix = unix_timestamp_to_datetime(t0_unix)
dt0_earliest_unix_ts = unix_timestamp_to_datetime(t0_earliest_unix_ts)
## OUTPUTS: dt0_unix, dt0_unix
dt0_unix, dt0_earliest_unix_ts

# (datetime.datetime(2026, 4, 27, 23, 42, 28, tzinfo=datetime.timezone.utc),
#  datetime.datetime(2026, 4, 29, 10, 38, 46, 298468, tzinfo=datetime.timezone.utc))
eeg_track_correction_delta: timedelta = (dt0_unix - dt0_earliest_unix_ts) # datetime.timedelta(days=-2, seconds=47021, microseconds=701532)

## find mice, call mice, mice are nice
eeg_track_correction_delta

## datetime.timedelta(days=-2, seconds=56444, microseconds=448690) - original offset
## datetime.timedelta(days=-1, seconds=64124, microseconds=812851) - Adding another session reduced the time, I guess because this knocked off one of the earliest sessions due to the fixed num active sessions

## I guess I lost the memories due to not taking the Mitrapizine last night. I also didn't take it for some days before. How sad. 



In [ ]:




eeg_track_correction_delta = FixupMisalignedData.extract_eeg_track_correction_delta(eeg_ds=eeg_ds)
eeg_track_correction_delta

## INPUTS: eeg_track_correction_delta
# eeg_track_correction_delta: datetime.timedelta = datetime.timedelta(days=-2, seconds=78719, microseconds=635839)
FixupMisalignedData.fix_all_timeline_tracks(timeline=timeline, eeg_track_correction_delta=eeg_track_correction_delta)


# Use the `eeg_track_correction_delta` to correct all the EEG-derived tracks:


In [ ]:
## Use the `eeg_track_correction_delta` to correct all the EEG-derived tracks:
# eeg_track_correction_delta

eeg_ds.earliest_unix_timestamp


In [ ]:
## apply it to all tracks in the timeline


In [ ]:
from pypho_timeline.widgets.simple_timeline_widget import FixupMisalignedData

eeg_track_correction_delta = FixupMisalignedData.extract_eeg_track_correction_delta(eeg_ds=eeg_ds)
eeg_track_correction_delta

## INPUTS: eeg_track_correction_delta
FixupMisalignedData.fix_all_timeline_tracks(timeline=timeline, eeg_track_correction_delta=eeg_track_correction_delta)



In [ ]:
[timeline.add_new_now_line_for_plot_item(widget.getRootPlotItem()) for widget in timeline.ui.matplotlib_view_widgets.values()]
timeline.update_now_lines()

In [ ]:
ratio_plot_item.setXRange(x0v, x1v, padding=0)
ratio_plot_item.setYRange(0, 1, padding=-0.02)



In [ ]:
float_to_datetime(detailed_td_ratio_df['t'].to_numpy(), reference_datetime=timeline.reference_datetime)


In [ ]:
td_ratio_track

In [ ]:
result

In [ ]:
from datetime import datetime, timezone

all_raws = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)
visible_t_start = float(eeg_ds.intervals_df["t_start"].min())
visible_t_end   = float(eeg_ds.intervals_df["t_start"].max() + eeg_ds.intervals_df["t_duration"].max())
print(f"Visible range: {datetime.fromtimestamp(visible_t_start, tz=timezone.utc)} -> {datetime.fromtimestamp(visible_t_end, tz=timezone.utc)}")

for i, r in enumerate(all_raws):
    md = r.info.get("meas_date")
    ts = md.timestamp() if md is not None else None
    in_range = (ts is not None) and (visible_t_start - 60 <= ts <= visible_t_end + 60)
    marker = "  <-- VISIBLE" if in_range else ""
    print(f"[{i}] meas_date={md} unix={ts} duration={r.times[-1]:.1f}s{marker}")

# ==================================================================================================================================================================================================================================================================================== #
# pre 2026-04-28 adjustment:                                                                                                                                                                                                                                                           #
# ==================================================================================================================================================================================================================================================================================== #
# Visible range: 2026-04-29 10:38:46.298468+00:00 -> 2026-04-30 01:19:33.430168+00:00
# [0] meas_date=2026-04-27 23:42:28+00:00 unix=1777333348.0 duration=908.0s
# [1] meas_date=2026-04-27 23:42:28+00:00 unix=1777333348.0 duration=908.0s
# [2] meas_date=2026-04-28 06:21:33+00:00 unix=1777357293.0 duration=52.6s
# [3] meas_date=2026-04-28 06:24:08+00:00 unix=1777357448.0 duration=4660.6s


In [ ]:
from datetime import datetime, timezone
import mne
import numpy as np
import pandas as pd
from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
from phopymnehelper.analysis.computations.eeg_registry import run_eeg_computations_graph, session_fingerprint_for_raw_or_path
from phopymnehelper.analysis.computations.specific.ADHD_sleep_intrusions import ThetaDeltaSleepIntrusionComputation, apply_adhd_sleep_intrusion_to_timeline

active_nearest_sess_idx: int = 2
eeg_raw = eeg_ds.get_sorted_and_extracted_raws(eeg_ds.raw_datasets_dict)[active_nearest_sess_idx]

_curr_compute_key: str = "theta_delta_sleep_intrusion"
eeg_comps_result = run_eeg_computations_graph(eeg_raw, session=session_fingerprint_for_raw_or_path(eeg_raw), goals=(_curr_compute_key,))
_curr_compute_result = eeg_comps_result[_curr_compute_key]
adhd_ctx = _curr_compute_result.copy()
out_adhd = adhd_ctx['out']
print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])
print(f'theta_delta_ratio: {out_adhd["theta_delta_ratio"]}')

## OUTPUTS: adhd_ctx, out_adhd

# out_adhd

In [ ]:
adhd_ctx

In [ ]:
out_adhd

In [ ]:
# self.apply_adhd_sleep_intrusion_to_timeline_plot_callback_fn(timeline=timeline)
# out_adhd['apply_adhd_sleep_intrusion_to_timeline_plot_callback_fn'](timeline)

result = eeg_comps_result["theta_delta_sleep_intrusion"]
print("session_mean_theta_delta", result["session_mean_theta_delta"], "valid", result["n_valid_windows"], "/", result["n_windows"])
apply_adhd_sleep_intrusion_to_timeline(timeline, result, eeg_name=eeg_name, eeg_ds=eeg_ds)


In [ ]:
# out_adhd = _curr_compute_result.copy()
# print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])
# print(f'theta_delta_ratio: {out_adhd["theta_delta_ratio"]}')

In [ ]:
# Synchronous run (blocks the kernel until done). For GUI threading see the next cell.
out_adhd = compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx.get("motion_df", None), window_sec=4.0, step_sec=1.0)
adhd_ctx["out"] = out_adhd
print("session_mean_theta_delta", out_adhd["session_mean_theta_delta"], "valid", out_adhd["n_valid_windows"], "/", out_adhd["n_windows"])


In [ ]:
adhd_ctx["t0"]

In [ ]:
import matplotlib.pyplot as plt

_x = adhd_ctx["t0"] + out_adhd["times"]
plt.figure(figsize=(10, 2.2))
plt.plot(_x, out_adhd["theta_delta_ratio"], color="tab:purple", linewidth=0.8)
plt.xlabel("Time (unix s)")
plt.ylabel("theta / delta")
plt.title("ADHD sleep intrusion series (NaN = motion/QC excluded)")
plt.tight_layout()
plt.show()

In [ ]:
apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx)

# Optional: run compute off the GUI thread, then marshal UI updates to the Qt main thread:
# def run_adhd_in_background():
#     from phopymnehelper.analysis.computations.specific import compute_theta_delta_sleep_intrusion_series
#     fut = ThreadPoolExecutor(max_workers=1).submit(lambda: compute_theta_delta_sleep_intrusion_series(adhd_ctx["raw"], motion_df=adhd_ctx["motion_df"], window_sec=4.0, step_sec=1.0))
#     def _done(f):
#         try:
#             adhd_ctx["out"] = f.result()
#         except Exception as exc:
#             print("ADHD compute failed:", exc)
#             return
#         QtCore.QTimer.singleShot(0, lambda: apply_adhd_sleep_intrusion_to_timeline(timeline, adhd_ctx))
#     fut.add_done_callback(_done)
# run_adhd_in_background()

In [ ]:
from phopymnehelper.analysis.computations.specific.bad_epochs import compute_bad_epochs_qc, apply_bad_epochs_overlays_to_timeline

out = compute_bad_epochs_qc(adhd_ctx["raw"], use_autoreject=True, copy_raw=True)
apply_bad_epochs_overlays_to_timeline(timeline, out, time_offset=t0)

In [ ]:
timeline.hide_extra_xaxis_labels_and_axes()

In [ ]:
timeline.get_all_track_names()


## Futher Timeline Widget/UI Customization

In [ ]:
from pypho_timeline.widgets.timeline_overview_strip import TimelineOverviewStrip
from pypho_timeline.EXTERNAL.pyqtgraph_extensions.graphicsObjects.CustomLinearRegionItem import CustomLinearRegionItem

strip: TimelineOverviewStrip = timeline.ui.timeline_overview_strip
region: CustomLinearRegionItem = strip._viewport_region
# strip._intervals_item.setEnabled(True)
# strip._intervals_item.movable = True
# region.setMovable(True) #.movable

In [ ]:
# strip

strip_active_window_float_ts = region.getRegion() # (1776323640.023324, 1776370487.5812337)
strip_active_window_float_ts # (1775008763.0, 1775065348.2999997)


In [ ]:
new_active_window_float_ts = (1776323640.023324, 1776370487.5812337)
new_active_window_float_ts


In [ ]:
region.setRegion(new_active_window_float_ts)


In [ ]:
# timeline.apply_active_window_from_plot_x(*strip_active_window_float_ts, block_signals=False)
# timeline.apply_active_window_from_plot_x(*new_active_window_float_ts, block_signals=True)
timeline.apply_active_window_from_plot_x(*(1775008763.0, 1775065348.2999997), block_signals=True)


In [ ]:
timeline.get_plot_view_range()

In [ ]:
# strip._intervals_item.data
latest_item_ts_float: float = strip._intervals_item.boundingRect().right() ## get right edge timestamp - 1775751404.0
curr_region_width_float: float = np.diff(region.getRegion())[0] # 43214.56095266342
curr_region_width_float

new_window_from_end = [(latest_item_ts_float-curr_region_width_float), latest_item_ts_float]
new_window_from_end

region.setRegion(new_window_from_end)
# strip.set_viewport(new_window_from_end)

In [ ]:
timeline.get_track_names_for_window_sync_group('primary')[0]

a_widget, _, _ = timeline.get_track_tuple(timeline.get_track_names_for_window_sync_group('primary')[0])
a_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
motion_widget, motion_track, motion_ds = timeline.get_track_tuple('MOTION_Epoc X Motion')
motion_widget.on_window_changed(start_t=1776323640.023324, end_t=1776370487.5812337)

In [ ]:
region.setAcceptHoverEvents(True)
region.setAcceptTouchEvents(True)

In [ ]:
# region.setRegion()
region.getRegion() # (1776323640.023324, 1776370487.5812337)



In [ ]:
timeline.total_data_end_time

In [ ]:
strip.setAcceptDrops(True)
strip.setInteractive(True)
strip.setMouseTracking(True)
strip.setMouseEnabled(True)

In [ ]:
vb = strip.getViewBox()
vb.setMouseEnabled(False, False)

In [ ]:
a_widget, a_renderer, a_ds = timeline.get_track_tuple('LOG_TextLogger')

In [ ]:
rendered_items = list(a_renderer.detail_graphics.values())[0]

rendered_text_items: list[pg.TextItem] = [v for v in rendered_items if isinstance(v, pg.TextItem)]
rendered_text_items


In [ ]:
for a_text_item in rendered_text_items:
    a_text_item.setAnchor([0.5, 0.5])
    a_text_item.set

In [ ]:
from pypho_timeline.rendering.datasources.specific import EEGTrackDatasource


a_ds: EEGTrackDatasource = timeline.track_datasources['EEG_Epoc X']
a_ds

In [ ]:
a_ds.intervals_df

In [ ]:

timeline = builder.build_from_xdf_file(demo_xdf_path)


In [ ]:
# timeline = main_all_modalities_from_xdf_file_example(demo_xdf_path)

In [ ]:
hasattr(motion_track, "channels_enabled")
motion_track.update_channel_visibility(channel_name='GyroX', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroY', is_visible=False)
motion_track.update_channel_visibility(channel_name='GyroZ', is_visible=False)

In [ ]:
motion_detail_renderer = motion_track.detail_renderer
motion_detail_renderer.normalize = False

motion_track.datasource.detailed_df
# motion_detail_renderer
# motion_detail_renderer.motion_df


In [ ]:
## INPUTS: timeline
# Get the "EEG Epoc X" track from the timeline
# Fallback to searching tracks by attribute, as there is no `.get_track_by_name` method
# eeg_track = timeline.get_track('EEG_Epoc X')
eeg_widget, eeg_track, eeg_ds = timeline.get_track_tuple('EEG_Epoc X')
detailed_eeg_df: pd.DataFrame = eeg_ds.detailed_df
# print(detailed_eeg_df.columns) # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4', 't']
detailed_eeg_df




In [ ]:
motion_ds._bad_intervals_unix_df = motion_ds._bad_intervals_unix_df.bad_motion_epochs_df.adding_unix_float_columns(inplace=False)
mask_bad_intervals_df: pd.DataFrame = motion_ds._bad_intervals_unix_df.copy()
mask_bad_intervals_df

In [ ]:
from phopymnehelper.helpers.dataframe_accessor_helpers import MaskedValidDataFrameAccessor

# mask_by_intervals

## find if each detailed_eeg_df['t'] is within the bad interval
# detailed_eeg_df['t']
# Efficiently see if each `detailed_eeg_df['t']` value falls within `mask_bad_intervals_df[['onset', 'onset_end']]` where all columns are datetime64[ns] 

# mask_bad_intervals_df[['onset', 'onset_end']]
# detailed_eeg_df['t'].dtypes # dtype('<M8[ns]')

detailed_eeg_df = detailed_eeg_df.masked_df.masking_by_intervals(mask_bad_intervals_df=mask_bad_intervals_df, time_col_name='t', bool_mask_column_name='is_bad_motion',
                                                            intervals_start_col_name='onset', intervals_end_col_name='onset_end')
detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'])
detailed_eeg_df


In [ ]:
detailed_eeg_df

In [ ]:
eeg_track.

In [ ]:


masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.get_masked(copy=True)

# masked_detailed_eeg_df: pd.DataFrame = detailed_eeg_df.masked_df.add_masking_column(mask_col='is_valid', value_cols=['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4'],
#                                                      copy=True)
masked_detailed_eeg_df

In [ ]:
masked_detailed_eeg_df.masked_df.get_masked()

In [ ]:
detailed_eeg_df = detailed_eeg_df.drop(columns=['is_bad_motion_right', 'is_bad_motion'], inplace=False)

In [ ]:

# ii = pd.IntervalIndex.from_arrays(
#     mask_bad_intervals_df["onset"],
#     mask_bad_intervals_df["onset_end"],
#     closed="both",  # or "left" / "right" / "neither"
# )
# in_bad = ii.contains(detailed_eeg_df["t"].values)


# import polars as pl

# points = pl.from_pandas(detailed_eeg_df).lazy()  # or scan_csv / DataFrame
# intervals = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

# hits = points.join_where(
#     intervals,
#     pl.col("t").is_between(pl.col("onset"), pl.col("onset_end")),
# )

# # timestamps that fall in at least one bad interval
# in_bad = (
#     hits.select("t")
#     .unique()
#     .with_columns(in_bad=pl.lit(True))
# )

# out = (
#     points.join(in_bad, on="t", how="left")
#     .with_columns(pl.col("in_bad").fill_null(False))
# )

# (pl.col("t") >= pl.col("onset")) & (pl.col("t") < pl.col("onset_end"))

import polars as pl

lf = pl.from_pandas(detailed_eeg_df).lazy()
iv = pl.from_pandas(mask_bad_intervals_df).select("onset", "onset_end").lazy()

bad_t = (
    lf.join(iv, how="cross")
    .filter((pl.col("t") >= pl.col("onset")) & (pl.col("t") <= pl.col("onset_end")))
    .select("t")
    .unique()
    .with_columns(pl.lit(True).alias("is_bad_motion"))
)

df = lf.join(bad_t, on="t", how="left").with_columns(
    pl.col("is_bad_motion").fill_null(False)
).collect()

detailed_eeg_df = df.to_pandas()  # if you need pandas again
detailed_eeg_df


In [ ]:
# eeg_track = timeline.get_track('MOTION_Epoc X Motion')

# if hasattr(timeline, "tracks"):
#     for tr in timeline.tracks:
#         # Try 'name' or 'track_name' attribute, fallback to 'datasource.name'
#         track_name = getattr(tr, "name", None) or getattr(tr, "track_name", None)
#         if track_name == "EEG Epoc X":
#             eeg_track = tr
#             break
#         elif hasattr(tr, "datasource") and hasattr(tr.datasource, "name"):
#             if tr.datasource.name == "EEG Epoc X":
#                 eeg_track = tr
#                 break
# eeg_track
eeg_track.channel_visibility

In [ ]:
hasattr(eeg_track, "channels_enabled")
eeg_track.update_channel_visibility(channel_name='GyroX', is_visible=False)


In [ ]:
channels_to_hide = [] # ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
channels_to_hide = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'FC6', 'F4', 'F8', 'AF4'] # , 'O1', 'O2', 'P8', 'T8'
for a_ch in channels_to_hide:
    eeg_track.update_channel_visibility(channel_name=a_ch, is_visible=False)
# eeg_track.update_channel_visibility(channel_name='P8', is_visible=False)


In [ ]:
print(list(eeg_track.channel_visibility.keys()))

In [ ]:
eeg_track.visible_intervals ## this should not be empty


In [ ]:


eeg_track._trigger_visibility_render()

In [ ]:

if eeg_track is not None and hasattr(eeg_track, "channel_visibility"):
    # If there's an attribute listing enabled channels, we'll disable half
    enabled_channels = eeg_track.channel_visibility
    # Otherwise fall back to all channel names if present
    if not enabled_channels and hasattr(eeg_track, "all_channel_names"):
        enabled_channels = eeg_track.all_channel_names
    if enabled_channels:
        # Compute half the channels (disable the second half)
        half = len(enabled_channels) // 2
        # Set only the first half as enabled
        eeg_track.channel_visibility = enabled_channels[:half]
        # If the track supports setting enabled/disabled channels via a method, use it
        if hasattr(eeg_track, 'set_enabled_channels'):
            eeg_track.set_enabled_channels(enabled_channels[:half])
        # Potentially trigger an update/redraw
        if hasattr(eeg_track, 'update_channel_visibility'):
            eeg_track.update_channel_visibility()
else:
    print("Could not find 'EEG Epoc X' track or it doesn't support channel control.")


# Timeline Widget from just Video Track

In [ ]:
enable_video_track: bool = True ## always true for run-video-only mode

In [ ]:
from pathlib import Path
from phopylslhelper.file_metadata_caching.manager import BaseFileMetadataManager
from phopylslhelper.file_metadata_caching.video_metadata import VideoMetadataParser
from phopylslhelper.file_metadata_caching.file_metadata import BaseFileMetadataParser

if enable_video_track:
    video_manager: BaseFileMetadataManager = BaseFileMetadataManager(parse_folders=[Path("M:/ScreenRecordings/EyeTrackerVR_Recordings"), Path("M:/ScreenRecordings/REC_continuous_video_recorder")],
                                                                    parsers={'video': VideoMetadataParser},)
    video_manager.metadata_df
    recent_videos = video_manager.get_most_recent_video_paths(max_num_videos=25)

In [ ]:
if enable_video_track:
    # Create the VideoTrackDatasource
    video_ds: VideoTrackDatasource = VideoTrackDatasource(video_paths=recent_videos)

    # Choose a name for the new video track
    video_track_name: str = "RecentVideosTrack"

    # Add the new video track to the existing timeline
    # timeline.add_video_track(video_track_name, video_ds)
    video_widget, root_graphics, plot_item, dock = timeline.add_video_track(track_name=video_track_name, video_datasource=video_ds,
                                                                                                                    enable_time_crosshair=True,
                                                                                                                )

# timeline.add_track(video_ds, name=video_track_name)


In [ ]:
from pypho_timeline.timeline_builder import TimelineBuilder
from pypho_timeline.rendering.graphics.track_renderer import TrackRenderer

builder = TimelineBuilder()

# Build a new timeline from just video track (no XDF file needed)
# Option 1: From video folder
# video_timeline = builder.build_from_video(video_folder_path=video_folder)

# Option 2: From list of video paths
video_timeline = builder.build_from_video(video_paths=recent_videos)

# Option 3: From existing VideoTrackDatasource
# video_timeline = builder.build_from_video(video_datasource=video_ds)
video_timeline.show()

In [ ]:
video_timeline.get_all_track_names()

In [ ]:
# video_widget
video_track_name = "VideoTrack"
video_widget, video_track_renderer, video_track_datasource = video_timeline.get_track_tuple(name=video_track_name)
video_widget

In [ ]:
# video_track_renderer.visible_intervals
a_brush = video_track_datasource.intervals_df['brush'].iloc[0]
a_brush.color() ## preview a QColor


In [ ]:
## Add a current datetime red line representing the current datetime (datetime.now())

video_widget.


In [ ]:
video_timeline.show()

In [ ]:
# Create a plot widget for this track
from pypho_timeline.core.synchronized_plot_mode import SynchronizedPlotMode


a_detail_renderer = video_ds.get_detail_renderer()

# The getOptionsPanel() method will be called by the dock when needed
# No need to set optionsPanel attribute if getOptionsPanel() is implemented

## if we provide a valid `optionsPanel: Optional[QWidget]` here on the widget the button will automatically show up on the dock
track_widget, a_root_graphics, a_plot_item, a_dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(
    name=video_ds.custom_datasource_name,
    dockSize=(500, 80),
    dockAddLocationOpts=['bottom'],
    sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA
)

assert a_detail_renderer is not None
track_widget.set_track_renderer(a_detail_renderer)
# Explicitly set the attribute (not just rely on getOptionsPanel())
track_widget.optionsPanel = track_widget.getOptionsPanel()

# Try to force dock to update button visibility
a_dock.updateWidgetsHaveOptionsPanel()

a_dock.update()  # May refresh the title bar
# Or if available:
if hasattr(a_dock, 'updateTitleBar') or hasattr(a_dock, 'refresh'):
    a_dock.updateTitleBar()  # or refresh()



# Set the plot to show the full time range
a_plot_item.setXRange(
    timeline.total_data_start_time, 
    timeline.total_data_end_time, 
    padding=0
)
a_plot_item.setYRange(0, 1, padding=0)
a_plot_item.setLabel('bottom', 'Time', units='s')
a_plot_item.setLabel('left', video_ds.custom_datasource_name)
a_plot_item.hideAxis('left')  # Hide Y-axis for cleaner look

# Add the track to the plot
a_track_name: str = video_ds.custom_datasource_name
timeline.add_track(
    video_ds,
    name=a_track_name,
    plot_item=a_plot_item
)

In [ ]:
video_track_datasource.intervals_df['video_file_path'].iloc[0]

In [ ]:
from deffcode import Sourcer
from deffcode import FFdecoder
DEFFCODE_AVAILABLE = True

In [ ]:
import pypho_timeline.rendering.datasources.specific.video

In [ ]:
from pypho_timeline.rendering.datasources.specific.video import VideoDeffcodeHelpers

video_path = Path(video_track_datasource.intervals_df['video_file_path'].iloc[0]).resolve()
assert video_path.exists()
primary_vid_metadata, vid_metadata = VideoDeffcodeHelpers.fetch_video_metadata_for_cache(a_video_file=video_path, debug_log_metadata=False)
source_duration_sec: float = primary_vid_metadata['source_duration_sec']
print(f'\tsource_duration_sec: {source_duration_sec}')
source_step_sec: float = (float(target_n_frames) / source_duration_sec)
print(f'\tsource_step_sec: {source_step_sec}')
# frame = VideoDeffcodeHelpers.fetch_video_metadata_and_thumbnail_for_cache(a_video_file=video_path, needs_metadata=False, save_output_thumbnail=False)
frame_offsets_sec = np.arange(start=0.0, stop=source_duration_sec, step=source_step_sec)
print(f'frame_offsets: {frame_offsets_sec}')
frames = VideoDeffcodeHelpers.fetch_video_thumbnails_for_cache(a_video_file=video_path, frame_offsets=frame_offsets_sec, save_output_thumbnail=False)
frames


In [ ]:

# Calculate timestamp relative to interval start
# frame_time_in_video = frame_idx / video_fps
timestamps.append(interval_start + frame_offsets_sec)

# 2026-04-15 - Export loaded sessions to EDF for analysis in other apps

In [ ]:
# 2026-04-15 - Export loaded sessions to EDF for analysis in other apps
from pathlib import Path
from typing import List
import mne

## INPUTS: xdf_to_exported_EDF_parent_export_path
assert xdf_to_exported_EDF_parent_export_path is not None
# xdf_to_exported_EDF_parent_export_path = sso.eeg_analyzed_parent_export_path
flat_raws: List[mne.io.Raw] = eeg_ds._flatten_raw_lists_from_dict(eeg_ds.raw_datasets_dict)
edf_export_parent_path: Path = xdf_to_exported_EDF_parent_export_path.resolve() # .joinpath("exported_EDF")
edf_export_parent_path.mkdir(exist_ok=True, parents=True)

exported_edf_paths: List[Path] = []
for raw_idx, a_raw in enumerate(flat_raws):
    source_stem = None
    raw_description = a_raw.info.get("description")
    if raw_description:
        try:
            source_stem = Path(str(raw_description)).stem
        except Exception:
            source_stem = None

    if not source_stem:
        raw_filenames = getattr(a_raw, "filenames", None)
        if raw_filenames and raw_filenames[0]:
            source_stem = Path(str(raw_filenames[0])).stem

    if not source_stem:
        source_stem = f"session_{raw_idx:03d}"

    curr_file_edf_path: Path = edf_export_parent_path.joinpath(f"{source_stem}.edf").resolve()
    exported_edf_paths.append(a_raw.save_to_edf(output_path=curr_file_edf_path))

print(f"Exported {len(exported_edf_paths)} EDF file(s) to: {edf_export_parent_path}")
exported_edf_paths

In [ ]:
from pathlib import Path
from phopymnehelper.xdf_files import XDFDataStreamAccessor, LabRecorderXDF

## INPUTS: loaded session raws in eeg_ds.raw_datasets_dict
post_processed_exports = {}
for a_sess_xdf_filename, eeg_raw_list in eeg_ds.raw_datasets_dict.items():
    session_raws_dict = {DataModalityType.EEG.value: eeg_raw_list}
    session_xdf_path = Path(str(a_sess_xdf_filename))
    eeg_raw, export_filepaths_dict = LabRecorderXDF.save_post_processed_to_fif(raws_dict=session_raws_dict, a_xdf_file=session_xdf_path, labRecorder_PostProcessed_path=sso.eeg_analyzed_parent_export_path.joinpath('LabRecorder_PostProcessed'))
    post_processed_exports[str(session_xdf_path)] = {'eeg_raw': eeg_raw, 'export_filepaths_dict': export_filepaths_dict}

post_processed_exports

In [ ]:
a_raw.save_to_edf(

# 2026-04-24 - Remove black inset border inside dock items

In [ ]:
from pypho_timeline.core.pyqtgraph_time_synchronized_widget import PyqtgraphTimeSynchronizedWidget
from pypho_timeline.widgets.custom_graphics_layout_widget import CustomGraphicsLayoutWidget


a_track_name: str = 'MOTION_Epoc X Motion'
# a_track_name: str = 'EEG_Epoc X'
a_renderer = timeline.track_renderers[a_track_name]
a_detail_renderer = a_renderer.detail_renderer # MotionPlotDetailRenderer 
a_ds = timeline.track_datasources[a_track_name]
# interval = a_ds.intervals_df
interval = a_ds.get_overview_intervals()
# interval = None
type(interval)
interval
a_renderer.visible_intervals
a_renderer.detail_graphics
a_renderer.overview_rects_item


dDisplayItem = timeline.ui.dynamic_docked_widget_container.find_display_dock(identifier=a_track_name) # Dock
a_widget: PyqtgraphTimeSynchronizedWidget = timeline.ui.matplotlib_view_widgets[a_track_name] # PyqtgraphTimeSynchronizedWidget 
a_root_graphics_layout_widget: CustomGraphicsLayoutWidget = a_widget.getRootGraphicsLayoutWidget()
a_plot_item = a_widget.getRootPlotItem()
a_plot_item


In [ ]:
a_plot_item.setAutoFillBackground(True)
a_plot_item.background_color 

In [ ]:
a_root_graphics_layout_widget.

# Capture tracks and config for each track for replication to a different window

In [ ]:
timeline.track_window_sync_groups

In [ ]:
timeline.track_datasources

In [ ]:
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow

a_main_window: MainTimelineWindow = timeline.window()
a_main_window.builder


In [ ]:
new_timeline = timeline.create_duplicate_timeline_widget()
secondary_window: MainTimelineWindow = MainTimelineWindow.init_with_timeline(timeline=new_timeline, builder=builder)
secondary_window

In [ ]:


new_timeline = timeline.create_duplicate_timeline_widget()
new_timeline

In [ ]:
new_timeline = timeline.create_child_window()
new_timeline

In [ ]:
from pypho_timeline.widgets.TimelineWindow.MainTimelineWindow import MainTimelineWindow

new_timeline = timeline.create_child_window()
new_timeline
secondary_window: MainTimelineWindow = MainTimelineWindow.init_with_timeline(timeline=new_timeline, builder=builder)
secondary_window


# Computed Dose Curve Tracks from parsing Logs

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource

# dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline, source_track_name='LOG_EventBoard')
# dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline)
dose_curve_ds = DoseTrackDatasource.build_dose_curve_track(timeline=timeline, timeline_builder=builder, source_track_name=['LOG_TextLogger', 'LOG_EventBoard'])

In [ ]:
# dose_curve_ds.get_detail_renderer()
dose_curve_ds.intervals_df
dose_curve_ds.total_df_start_end_times
dose_curve_ds.recordSeries_df
dose_curve_ds.complete_curve_df


In [ ]:
"""
[2026-04-25 01:23:01] EventBoard: Dose 1.0+ (DOSE_AMPH_30mg)
[2026-04-25 01:44:29] [TRANSCRIBED] Thank  you.
[2026-04-25 01:44:36] [TRANSCRIBED] you.
[2026-04-25 01:45:08] [TRANSCRIBED] Thanks  for  watching!
[2026-04-25 01:45:15] [TRANSCRIBED] Thanks  for  watching!
[2026-04-25 03:52:27] EventBoard: Dose 1.0+ (DOSE_AMPH_30mg)
[2026-04-25 03:52:30] [TRANSCRIBED] 1 .0.
[2026-04-25 03:52:35] [TRANSCRIBED] .0
[2026-04-25 03:52:41] [TRANSCRIBED] Plus.
"""
from dose_analysis_python.FileImportExport.DoseImporter import DoseNoteFragmentParser
from phopymnehelper.MNE_helpers import MNEHelpers
from phopylslhelper.datetime_helpers import to_display_timezone, unix_timestamp_to_datetime, float_to_datetime


all_track_names = timeline.get_all_track_names()
print(f"all_track_names: {all_track_names}")
txt_log_widget, txt_log_renderer, txt_log_ds = timeline.get_track_tuple('LOG_TextLogger')
txt_log_ds

detailed_txt_log_df: pd.DataFrame = deepcopy(txt_log_ds.detailed_df).reset_index(drop=True) #.set_index('t')

if 'dt' not in detailed_txt_log_df.columns:
    detailed_txt_log_df['dt'] = float_to_datetime(detailed_txt_log_df['t'].to_numpy(), reference_datetime=timeline.reference_datetime)

detailed_txt_log_df = detailed_txt_log_df.set_index('dt', drop=False, inplace=False)
## OUTPUTS: detailed_txt_log_df
detailed_txt_log_df

In [ ]:
def parse_message_logs_to_dose_events(detailed_txt_log_df):
    ## Parse message logs for dose-like entries
    ## INPUTS: detailed_txt_log_df
    detailed_txt_log_df['msg']
    detailed_txt_log_df 


    # r'\d+\s*\.?\s*\d*\s*plus'

    # dose_event_pattern = r'\d+\s*\.?\s*\d*\s*plus'
    # matching_rows = detailed_txt_log_df[detailed_txt_log_df['msg'].astype(str).str.contains(dose_event_pattern, regex=True, na=False)]
    # matching_rows

    dose_event_pattern = r'(\d+\s*\.?\s*\d*)\s*plus'

    matched_with_quantity = (
        detailed_txt_log_df.loc[detailed_txt_log_df['msg'].astype(str).str.contains(dose_event_pattern, regex=True, na=False), ['msg']]
        .assign(quantity=lambda df: df['msg'].astype(str).str.extract(dose_event_pattern, expand=False).str.replace(r'\s+', '', regex=True))
    )

    matched_with_quantity

    # matched_with_quantity['dt']

    # a_parsed_dose_record = DoseNoteFragmentParser.parse_numeric_dose([a_parsed_dose])[0]

    parsed_dose_records_dict: Dict[pd.Timestamp, Dict] = dict(zip(matched_with_quantity.index.to_list(), DoseNoteFragmentParser.parse_numeric_dose(matched_with_quantity['quantity'].to_list())))
    parsed_dose_records_dict
    ## OUTPUTS: parsed_dose_records_dict


    ## build records
    default_med_name: str='AMPH'
    recordSeries_df: pd.DataFrame = pd.DataFrame({'recordDoseDate': list(parsed_dose_records_dict.keys()), 
                                                'recordDoseValue': [a_dose_record_dict['value'] for k, a_dose_record_dict in parsed_dose_records_dict.items()],
                                                'modifier': [a_dose_record_dict.get('modifier', '') for k, a_dose_record_dict in parsed_dose_records_dict.items()],
                            })
    recordSeries_df = recordSeries_df.set_index('recordDoseDate', drop=True, inplace=False)
    # recordSeries.index = pd.DatetimeIndex(recordSeries.index).tz_localize("US/Eastern")
    recordSeries_df.index = pd.DatetimeIndex(recordSeries_df.index).tz_convert("US/Eastern")
    recordSeries_df['medication'] = default_med_name
    ## OUTPUTS: recordSeries_df
    return recordSeries_df

## INPUTS: detailed_txt_log_df
recordSeries_df = parse_message_logs_to_dose_events(detailed_txt_log_df)
recordSeries_df

In [ ]:
from datetime import datetime, timedelta
from dose_analysis_python.Helpers.quantization import Quanta, ComputationTimeBlock

## INPUTS: timeline, recordSeries_df
# def build_dose_curve_track(timeline, recordSeries_df):
## compute specific datetime windows by passing the initial/final state vector to the calculation
start_date: datetime = timeline.total_data_start_time # datetime(2026, 4, 13)
# end_date: Optional[datetime] = None
end_date: Optional[datetime] = timeline.total_data_end_time

computation_blocks, complete_curve_df = ComputationTimeBlock.init_from_start_end_date(parsed_record_df=recordSeries_df, start_date=start_date, end_date=end_date)
## OUTPUTS: computation_blocks, complete_curve_df
# print(list(complete_curve_df.columns)) # ['t_h', 'AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc', 'compute_block_idx', 't']
complete_curve_df

In [ ]:
import pandas as pd
from pypho_timeline import SynchronizedPlotMode
from pypho_timeline.rendering.helpers import ChannelNormalizationMode
from pypho_timeline.rendering.datasources.track_datasource import IntervalProvidingTrackDatasource
from pypho_timeline.rendering.detail_renderers.generic_plot_renderer import DataframePlotDetailRenderer


def build_dose_curve_track(timeline, complete_curve_df, track_name = 'DOSE_CURVES_Computed'):
    ## INPUTS: complete_curve_df
    ## time_column_names = ['t_h', 't']
    curve_channel_names = [col for col in ['AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc'] if col in complete_curve_df.columns]
    curve_df = complete_curve_df[['t'] + curve_channel_names].copy()
    interval_start = curve_df['t'].min()
    interval_duration_seconds = (curve_df['t'].max() - interval_start).total_seconds()
    intervals_df = pd.DataFrame({'t_start': [interval_start], 't_duration': [interval_duration_seconds]})
    curve_renderer = DataframePlotDetailRenderer(channel_names=curve_channel_names, normalize=True, 
                                                normalization_mode_dict={
            ('AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf'): ChannelNormalizationMode.GROUPMINMAXRANGE,
            ('DA_str', 'DA_pfc'): ChannelNormalizationMode.GROUPMINMAXRANGE,
            ('NE_pfc',): ChannelNormalizationMode.GROUPMINMAXRANGE,
        }, fallback_normalization_mode = ChannelNormalizationMode.INDIVIDUAL,
        pen_width=1.5,
    )

    dose_curve_ds = IntervalProvidingTrackDatasource(intervals_df=intervals_df, detailed_df=curve_df, custom_datasource_name=track_name, detail_renderer=curve_renderer, enable_downsampling=False)

    if track_name in timeline.get_all_track_names():
        print(f"Track '{track_name}' already exists; skipping add.")
    else:
        track_widget, root_graphics, plot_item, dock = timeline.add_new_embedded_pyqtgraph_render_plot_widget(name=track_name, dockSize=(500, 120), dockAddLocationOpts=['bottom'], sync_mode=SynchronizedPlotMode.TO_GLOBAL_DATA)
        timeline.add_track(dose_curve_ds, name=track_name, plot_item=plot_item)

    
    return dose_curve_ds, curve_renderer



In [ ]:
dose_curve_ds.cl

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource


# complete_curve_df, recordSeries_df = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline=timeline)

dose_curve_ds = DoseTrackDatasource.init_from_timeline_text_log_tracks(timeline=timeline)
dose_curve_ds
# dose_curve_ds, curve_renderer = DoseTrackDatasource.build_dose_curve_track(timeline, recordSeries_df, track_name = 'DOSE_CURVES_Computed')


In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DoseTrackDatasource

dose_curve_ds = DoseTrackDatasource.build_dose_curve_track(
    timeline=timeline,
    timeline_builder=builder,
    source_track_name='LOG_TextLogger', #'LOG_EventBoard'   # or 'LOG_TextLogger' for other setups
)

In [ ]:
from pypho_timeline.rendering.datasources.specific.dose import DosePlotDetailRenderer

dose_curve_renderer = DosePlotDetailRenderer(text_color='orange', text_size=10, channel_names=modality_channels_dict['LOG'])

datasource = IntervalProvidingTrackDatasource.from_multiple_sources(intervals_dfs=all_intervals_dfs, detailed_dfs=all_detailed_dfs, custom_datasource_name=f"LOG_{stream_name}", detail_renderer=a_detail_renderer, enable_downsampling=False)



In [ ]:
print(list(complete_curve_df.columns)) # ['t_h', 'AMPH_gut', 'AMPH_blood', 'AMPH_brain', 'AMPH_ecf', 'DA_str', 'NE_pfc', 'DA_pfc', 'compute_block_idx', 't']


In [ ]:
# from pypho_timeline.utils.datetime_helpers import float_to_datetime, unix_timestamp_to_datetime, get_reference_datetime_from_xdf_header, to_display_timezone
from phopylslhelper.datetime_helpers import to_display_timezone, unix_timestamp_to_datetime, float_to_datetime

detailed_txt_log_df['t'].to_numpy()

out_dts = unix_timestamp_to_datetime(detailed_txt_log_df['t'].to_numpy())
out_dts
[to_display_timezone(v) for v in out_dts]



In [ ]:


to_display_timezone(np.array(unix_timestamp_to_datetime(detailed_txt_log_df['t'].to_numpy())))

.astimezone(DISPLAY_TIMEZONE)

# - [ ] Fix the red line or die trying

In [ ]:
[timeline.add_new_now_line_for_plot_item(widget.getRootPlotItem()) for widget in timeline.ui.matplotlib_view_widgets.values()]
timeline.update_now_lines()

In [ ]:
timeline.plots_data['now_lines'].now_dt

In [ ]:
for plot_item, vline in timeline.plots.now_lines.now_line_items.items():
    if (plot_item is not None) and (vline is not None):
        vline.setPos(timeline.plots_data['now_lines'].now_timestamp) ## moves the item